# ThermalTwin — House Cooling / Cross Ventilation Model

This notebook demonstrates `HouseCoolingCrossVentilationPredictor` (`models/house_cooling_cross_ventilation.py`), a physics-based model of natural cross ventilation and the resulting house cooling.

Unlike the data-fitted models in the other notebooks, this model does not learn parameters from measured temperature data — the house/window geometry and thermal mass are physical properties of the building, supplied directly. The notebook:

1. Computes the natural ventilation airflow through the windows
2. Computes ACH (air changes per hour)
3. Simulates the house cooling down
4. Compares several fan sizes
5. Plots the results

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from models.house_cooling_cross_ventilation import HouseCoolingCrossVentilationPredictor

## 1. House Parameters

Net open window area on each of the two cross-ventilation sides, house geometry, and the discharge coefficient (Cd) of the openings. Typical Cd values are 0.5 (poor), 0.6 (normal), 0.7 (good); the value below is empirically tuned for this house.

In [ ]:
FLOOR_AREA_M2 = 57
CEILING_HEIGHT_M = 2.6

# net open window areas
WINDOW_AREA_SIDE_A_M2 = 1030 * 700 / 1000**2
WINDOW_AREA_SIDE_B_M2 = 1400 * 700 / 1000**2 + 1400 * 1100 / 1000**2

DISCHARGE_COEFFICIENT = 0.015

# thermal mass factor: higher -> house cools more slowly
# 1-2 light-weight house, 3-5 average apartment, 5-8 heavy-weight house
THERMAL_MASS_FACTOR = 8

FAN_EFFICIENCY = 0.65

model = HouseCoolingCrossVentilationPredictor(
    floor_area_m2=FLOOR_AREA_M2,
    ceiling_height_m=CEILING_HEIGHT_M,
    window_area_side_a_m2=WINDOW_AREA_SIDE_A_M2,
    window_area_side_b_m2=WINDOW_AREA_SIDE_B_M2,
    discharge_coefficient=DISCHARGE_COEFFICIENT,
    thermal_mass_factor=THERMAL_MASS_FACTOR,
    fan_efficiency=FAN_EFFICIENCY,
)

# fit() is a no-op for this model (geometry is given, not learned),
# but it is still required before predict()/simulate() to match the
# BasePredictor interface used across the project.
model.fit()

## 2. Outdoor Conditions and Natural Ventilation

In [ ]:
WIND_SPEED_M_S = 3
OUTSIDE_TEMP_C = 20
INSIDE_TEMP_C = 24

natural_flow = model.natural_flow_m3_h(WIND_SPEED_M_S)
natural_ach = model.air_changes_per_hour(WIND_SPEED_M_S, fan_capacity_m3_h=0)

print("================================================")
print("HOUSE COOLING MODEL")
print("================================================")
print(f"House volume: {model.house_volume_m3:.1f} m3")
print(f"Effective window area: {model.effective_window_area_m2:.2f} m2")
print()
print("--- Natural ventilation ---")
print(f"Natural airflow: {natural_flow:.0f} m3/h")
print(f"Natural ACH: {natural_ach:.1f}")
print(f"Air refreshed every {60 / natural_ach:.1f} minutes")

## 3. Compare Fan Capacities

`0 m3/h` means natural ventilation only. Each scenario simulates the same fixed outdoor temperature and wind speed over `SIMULATION_HOURS`, using the model's cooling ODE `dT/dt = -k(T - T_out)` with `k = ACH / thermal_mass_factor`.

In [ ]:
FAN_CAPACITIES_M3_H = [0, 1000, 2000, 3000, 5000]

SIMULATION_HOURS = 6
TIME_STEP_MINUTES = 1
DT_SECONDS = TIME_STEP_MINUTES * 60

results = model.compare_fan_capacities(
    outside_temp_C=OUTSIDE_TEMP_C,
    T0=INSIDE_TEMP_C,
    wind_speed_m_s=WIND_SPEED_M_S,
    fan_capacities_m3_h=FAN_CAPACITIES_M3_H,
    duration_hours=SIMULATION_HOURS,
    dt=DT_SECONDS,
)

steps = int(SIMULATION_HOURS * 3600 / DT_SECONDS)
times_h = np.arange(steps) * (DT_SECONDS / 3600)

for fan_nominal, result in results.items():
    print(f"Fan nominal: {fan_nominal:.0f} m3/h")
    print(f"  Total airflow: {result['flow_m3_h']:.0f} m3/h")
    print(f"  ACH: {result['ach']:.1f}")
    print(f"  Temperature after {SIMULATION_HOURS}h: {result['temps'][-1]:.1f} C")

## 4. Temperature Decay vs Fan Capacity

In [ ]:
plt.figure(figsize=(12, 6))

for fan_nominal, result in results.items():
    label = f"{fan_nominal} m3/h (ACH={result['ach']:.1f})"
    plt.plot(times_h, result["temps"], linewidth=3, label=label)

plt.axhline(OUTSIDE_TEMP_C, linestyle="--", color="black", alpha=0.7, label="Outdoor temperature")

plt.xlabel("Time [hours]")
plt.ylabel("Indoor temperature [\u00b0C]")
plt.title("House cooling vs fan capacity")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Air Changes per Hour

In [ ]:
achs = [results[f]["ach"] for f in FAN_CAPACITIES_M3_H]

plt.figure(figsize=(10, 5))
plt.bar([str(f) for f in FAN_CAPACITIES_M3_H], achs)
plt.xlabel("Fan capacity [m3/h]")
plt.ylabel("ACH")
plt.title("Air Changes per Hour")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## 6. Time to Reach a Target Temperature

In [ ]:
TARGET_TEMP_C = 20

times_to_target = [
    HouseCoolingCrossVentilationPredictor.time_to_reach_temperature(
        results[f]["temps"], TARGET_TEMP_C, dt=DT_SECONDS
    )
    for f in FAN_CAPACITIES_M3_H
]

plt.figure(figsize=(10, 5))
plt.bar([str(f) for f in FAN_CAPACITIES_M3_H], times_to_target)
plt.xlabel("Fan capacity [m3/h]")
plt.ylabel(f"Time to {TARGET_TEMP_C}\u00b0C [hours]")
plt.title(f"Time for the house to cool down to {TARGET_TEMP_C}\u00b0C")
plt.grid(axis="y")
plt.tight_layout()
plt.show()